In [1]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()  
print("Spark aktif:", spark.version)

Spark aktif: 3.5.9


In [5]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline

idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline Logistic Regression terdefinisi (6 tahap).")

Pipeline Logistic Regression terdefinisi (6 tahap).


In [12]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Training: {train.count()} | Testing: {test.count()}")

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1  = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr, f1_lr, auc_lr = ev_acc.evaluate(pred_lr), ev_f1.evaluate(pred_lr), ev_auc.evaluate(pred_lr)

print("=== Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Training: 2411 | Testing: 589
=== Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [13]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=42)
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

acc_rf, f1_rf, auc_rf = ev_acc.evaluate(pred_rf), ev_f1.evaluate(pred_rf), ev_auc.evaluate(pred_rf)

print(f"{'Model':<22}{'Accuracy':>10}{'F1':>10}{'AUC':>10}")
print(f"{'Logistic Regression':<22}{acc_lr:>10.4f}{f1_lr:>10.4f}{auc_lr:>10.4f}")
print(f"{'Random Forest':<22}{acc_rf:>10.4f}{f1_rf:>10.4f}{auc_rf:>10.4f}")

Model                   Accuracy        F1       AUC
Logistic Regression       0.7929    0.7926    0.8650
Random Forest             0.7963    0.7958    0.8474


In [14]:
import pandas as pd
from pyspark.sql.functions import col, count, when, round as sround

# Pilih model terbaik berdasarkan AUC tertinggi
if auc_rf > auc_lr:
    nama_terbaik, pred_terbaik, model_terbaik = "Random Forest", pred_rf, model_rf
else:
    nama_terbaik, pred_terbaik, model_terbaik = "Logistic Regression", pred_lr, model_lr
print("Model terbaik (AUC tertinggi):", nama_terbaik)

# 10 baris yang salah diprediksi
salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print(f"Salah prediksi: {salah.count()} dari {pred_terbaik.count()}")

salah.select("nama_menu", "cabang", "waktu_pesan", "qty",
             "waktu_tunggu_menit", "ulasan_positif", "prediction").show(10)

# Pola: persentase salah per waktu tunggu
pred_terbaik.groupBy("waktu_tunggu_menit") \
    .agg(count("*").alias("total"),
         count(when(col("ulasan_positif") != col("prediction"), 1)).alias("salah")) \
    .withColumn("persen_salah", sround(col("salah") / col("total") * 100, 1)) \
    .orderBy("waktu_tunggu_menit").show(25)

Model terbaik (AUC tertinggi): Logistic Regression
Salah prediksi: 122 dari 589
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|  cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|      Malam|  1|                15|             1|       0.0|
|Americano|Magelang|      Siang|  2|                22|             1|       0.0|
|Americano|Magelang|       Sore|  1|                 7|             0|       1.0|
|Americano|Magelang|       Sore|  2|                11|             0|       1.0|
|Americano|Magelang|       Sore|  3|                 5|             0|       1.0|
|Americano|Magelang|       Sore|  3|                13|             1|       0.0|
|Americano|Magelang|       Sore|  5|                 6|             0|       1.0|
|Americano|Magelang|       Sore|  5|                15|             1|       0.0|
|Americano|Semaran

Kesimpulan 

Model terbaik adalah Logistic Regression angka .Fitur paling berpengaruh adalah waktu tunggu pesanan; menu, cabang, dan waktu pesan hampir tidak berpengaruh. Kesalahan prediksi paling banyak terjadi pada waktu tunggu sekitar [rentang, misalnya 10-14 menit]. Rekomendasi untuk manajemen Kopi Nusantara: percepat layanan dengan menambah barista pada jam sibuk, tetapkan batas waktu tunggu maksimal [angka, misalnya 10] menit di semua cabang, dan beri peringatan dini kepada staf saat pesanan melewati batas tersebut.